# 2 · Entrenamiento y evaluación

> **Tipo de ML:** `{{ cookiecutter.ml_type }}`


## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from {{ cookiecutter.project_module_name }}.utils.paths import PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR

{% if cookiecutter.ml_type == 'supervisado' %}
from {{ cookiecutter.project_module_name }}.models.train_model import train_models, load_models
from {{ cookiecutter.project_module_name }}.models.predict_model import evaluate_models, predict_new, DECISION_THRESHOLD
from {{ cookiecutter.project_module_name }}.visualization.visualize import plot_feature_importance
{% elif cookiecutter.ml_type == 'no_supervisado' %}
from {{ cookiecutter.project_module_name }}.models.train_model import train_models, find_optimal_k
from {{ cookiecutter.project_module_name }}.models.predict_model import evaluate_models, plot_dendrogram
from {{ cookiecutter.project_module_name }}.visualization.visualize import (
    plot_elbow_and_silhouette, plot_dendrogram as viz_dendrogram, plot_pca_variance,
)
{% elif cookiecutter.ml_type == 'redes_neuronales' %}
from {{ cookiecutter.project_module_name }}.models.train_model import train_models, load_model
from {{ cookiecutter.project_module_name }}.models.predict_model import evaluate_models
{% endif %}


## 2. Cargar datos procesados


In [ ]:
{% if cookiecutter.ml_type in ['supervisado', 'redes_neuronales'] %}
X_train = pd.read_csv(PROCESSED_DATA_DIR / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DATA_DIR / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DATA_DIR / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED_DATA_DIR / 'y_test.csv').squeeze()
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
{% elif cookiecutter.ml_type == 'no_supervisado' %}
import numpy as np
X = pd.read_csv(PROCESSED_DATA_DIR / 'X_processed.csv').values
print(f'Shape: {X.shape}')
{% endif %}


## 3. Selección del número de clusters

{% if cookiecutter.ml_type == 'no_supervisado' %}
Usamos tres métodos complementarios para elegir k:
- **Dendrograma**: línea vertical más larga sin cruzar ninguna horizontal → cuenta las ramas.
- **Método del codo**: busca el punto donde la inercia deja de bajar bruscamente.
- **Silhouette Score**: maximizar (rango [-1, 1]; más alto es mejor).
{% endif %}


In [ ]:
{% if cookiecutter.ml_type == 'no_supervisado' %}
# 1. Dendrograma (clustering jerarquico aglomerativo)
viz_dendrogram(X, method='ward')
# Tras ver el dendrograma, descomenta la siguiente linea con el umbral de corte:
# viz_dendrogram(X, method='ward', color_threshold=50)

# 2. Varianza explicada PCA (util si hay muchas dimensiones)
plot_pca_variance(X)

# 3. Elbow + Silhouette
metrics = find_optimal_k(X, k_range=range(2, 11))
plot_elbow_and_silhouette(metrics)
{% else %}
pass  # No aplica para este tipo de ML
{% endif %}


## 4. Entrenar modelos


In [ ]:
{% if cookiecutter.ml_type == 'supervisado' %}
models = train_models(X_train, y_train, tune_knn=True, cv_evaluate=True)
{% elif cookiecutter.ml_type == 'no_supervisado' %}
N_CLUSTERS = 3   # ajusta tras ver el dendrograma y el metodo del codo
models = train_models(X, n_clusters=N_CLUSTERS)
{% elif cookiecutter.ml_type == 'redes_neuronales' %}
input_dim  = X_train.shape[1]
output_dim = y_train.nunique()
models = train_models(
    X_train, y_train,
    input_dim=input_dim, output_dim=output_dim,
    epochs=50, batch_size=32, checkpoint_every=10,
)
{% endif %}


## 5. Evaluar


In [ ]:
{% if cookiecutter.ml_type == 'supervisado' %}
threshold = DECISION_THRESHOLD   # modifica si el dataset esta desbalanceado
df_results = evaluate_models(
    models, X_train, y_train, X_test, y_test, threshold=threshold
)
df_results
{% elif cookiecutter.ml_type == 'no_supervisado' %}
evaluate_models(models, X)
{% elif cookiecutter.ml_type == 'redes_neuronales' %}
num_classes = y_test.nunique()
evaluate_models(models, X_test, y_test, num_classes=num_classes)
{% endif %}


## 6. Importancia de variables


In [ ]:
{% if cookiecutter.ml_type == 'supervisado' %}
feature_names = X_train.columns.tolist()
plot_feature_importance(models, feature_names)
from IPython.display import Image
Image(FIGURES_DIR / 'feature_importance.png')
{% else %}
pass  # No disponible para este tipo de ML
{% endif %}


## 7. Prediccion sobre nuevos datos


In [ ]:
{% if cookiecutter.ml_type == 'supervisado' %}
best_name = df_results.sort_values('Acc_test', ascending=False).iloc[0]['Modelo']
best_model = models[best_name]
print(f'Mejor modelo: {best_name}')

# Descomenta para predecir sobre nuevos datos:
# from {{ cookiecutter.project_module_name }}.features.build_features import process_input
# X_new = process_input(df_nuevos)
# preds, probs = predict_new(best_model, X_new, threshold=threshold)
{% elif cookiecutter.ml_type == 'no_supervisado' %}
# Prediccion sobre nuevos datos (KMeans tiene .predict, AgglomerativeClustering no):
# model = joblib.load(MODELS_DIR / 'KMeans.joblib')
# labels = model.predict(X_new)
{% elif cookiecutter.ml_type == 'redes_neuronales' %}
# model = load_model(input_dim=input_dim, output_dim=output_dim)
# import torch
# X_tensor = torch.tensor(X_new.values, dtype=torch.float32)
# preds = torch.argmax(model(X_tensor), dim=1)
{% endif %}
